In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==========================================
# 1. LOAD DATA & PREPARE 3D SEQUENCES
# ==========================================
df_dl_v3 = pd.read_csv("../data/processed/demand_features_v3.csv", parse_dates=["date"])

feature_cols_v3 = ['year', 'month', 'day_of_week', 'is_weekend', 'is_back_to_school', 'is_holiday_season', 
                   'lag_1', 'lag_7', 'rolling_mean_7', 'rolling_mean_30', 'is_promo', 'is_stockout']

SEQ_LENGTH = 14 # Look back 14 days to predict the next day

all_X_train, all_y_train = [], []
all_X_test, all_y_test, all_actual_test, all_lag_test = [], [], [], []

for sku in df_dl_v3['sku_id'].unique():
    sku_df = df_dl_v3[df_dl_v3['sku_id'] == sku].reset_index(drop=True)
    
    # Neural Networks require scaled features to prevent exploding gradients
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(sku_df[feature_cols_v3])
    
    target = sku_df['units_sold_diff'].values
    actual = sku_df['units_sold'].values
    lag_1 = sku_df['lag_1'].values
    
    X, y, actuals, lags = [], [], [], []
    
    # Create sliding windows of 14 days
    for i in range(len(sku_df) - SEQ_LENGTH):
        X.append(scaled_features[i : i + SEQ_LENGTH])
        y.append(target[i + SEQ_LENGTH])
        actuals.append(actual[i + SEQ_LENGTH])
        lags.append(lag_1[i + SEQ_LENGTH])
        
    X, y, actuals, lags = np.array(X), np.array(y), np.array(actuals), np.array(lags)
    
    # 80/20 chronological split per SKU
    split_idx = int(len(X) * 0.8)
    
    all_X_train.append(X[:split_idx])
    all_y_train.append(y[:split_idx])
    all_X_test.append(X[split_idx:])
    all_y_test.append(y[split_idx:])
    all_actual_test.append(actuals[split_idx:])
    all_lag_test.append(lags[split_idx:])

# Convert to PyTorch Tensors
X_train = torch.tensor(np.concatenate(all_X_train), dtype=torch.float32)
y_train = torch.tensor(np.concatenate(all_y_train), dtype=torch.float32)
X_test = torch.tensor(np.concatenate(all_X_test), dtype=torch.float32)

actual_test = np.concatenate(all_actual_test)
lag_test = np.concatenate(all_lag_test)

# Create DataLoader for batching
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)


# ==========================================
# 2. DEFINE THE TRANSFORMER ARCHITECTURE
# ==========================================
class TimeSeriesTransformer(nn.Module):
    def __init__(self, num_features, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        # 1. Project input tabular features to a higher-dimensional embedding (d_model)
        self.input_projection = nn.Linear(num_features, d_model)
        
        # 2. Transformer Encoder Layer (Learns relationships across the 14-day sequence)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dropout=dropout, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 3. Final output layer predicting the differenced target value
        self.decoder = nn.Linear(d_model, 1)
        
    def forward(self, x):
        # x shape: (batch_size, seq_length, num_features)
        x = self.input_projection(x)
        x = self.transformer(x)
        
        # We only care about forecasting the next day, so we take the output at the last time step
        last_step_output = x[:, -1, :] 
        out = self.decoder(last_step_output)
        return out.squeeze()


# ==========================================
# 3. TRAIN THE MODEL
# ==========================================
model_tft = TimeSeriesTransformer(num_features=len(feature_cols_v3))
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_tft.parameters(), lr=0.001)

epochs = 20
print("Starting Training...")
model_tft.train()

for epoch in range(epochs):
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model_tft(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f}")


# ==========================================
# 4. EVALUATE PREDICTIONS
# ==========================================
model_tft.eval()
with torch.no_grad():
    # Model predicts the difference
    test_preds_diff = model_tft(X_test).numpy()
    
# Reconstruct actual values
preds_actual_tft = lag_test + test_preds_diff

# Calculate Metrics
rmse_tft = np.sqrt(mean_squared_error(actual_test, preds_actual_tft))
mae_tft = mean_absolute_error(actual_test, preds_actual_tft)
mape_tft = np.mean(np.abs((actual_test - preds_actual_tft) / actual_test)) * 100
r2_tft = r2_score(actual_test, preds_actual_tft)

print("\n--- PyTorch Transformer Performance ---")
print(f"RMSE: {rmse_tft:.3f}")
print(f"MAE:  {mae_tft:.3f}")
print(f"MAPE: {mape_tft:.3f}%")
print(f"R2:   {r2_tft:.3f}")

Starting Training...
Epoch 5/20 | Loss: 5942.5465
Epoch 10/20 | Loss: 5346.6078
Epoch 15/20 | Loss: 4854.8126
Epoch 20/20 | Loss: 4397.9075

--- PyTorch Transformer Performance ---
RMSE: 93.036
MAE:  48.726
MAPE: 19.743%
R2:   0.429
